Documentation: [readme.md](readme.md). The first code cell switches the process **cwd** to **`deploy/`** so `utils.py` imports work whether you start Jupyter from the **repo root** or from **`deploy/`**.

Environment: from the **repository root**, run **`uv sync`** (creates **`.venv/`** at the repo root) then **`uv run jupyter notebook deploy/deploy.ipynb`** (see readme).

**Firmware `.bin`**: store under **`deploy/bin/`** or rely on auto-download inside **`flash_with_esptool`** / **`run_full_flash`** (see readme).

Deployments are **Step 1–3** below; step code is **uncommented** so you can run each cell as needed. **Run all** will execute Step 1, then 2, then 3 in order (often wrong for one device: Step 3 is usually for **existing** boards only, not right after Step 2 on a new install). Run only the cells that match your task.

## Configuration

Run the **setup** cell below, then **Serial port** to list USB devices and optionally pick one for **`CFG.serial_port`** (default **`/dev/tty.usbmodem101`**).

You can also edit **`CFG`** directly: `circuitpy_root`, `settings_slot` (for Step 3), and optional **`circuitpython_bin`** / **`circuitpython_board_id`** / **`circuitpython_locale`** / **`circuitpython_download_fallback_url`**.

In [1]:
import sys
from pathlib import Path

# So ``import notebook_env`` works from repo root or from deploy/
_repo = Path.cwd().resolve()
_deploy_candidate = _repo if (_repo / "utils.py").is_file() else _repo / "deploy"
if str(_deploy_candidate) not in sys.path:
    sys.path.insert(0, str(_deploy_candidate))

import notebook_env

notebook_env.activate()

from utils import (
    DeployConfig,
    copy_firmware_to_circuitpy,
    flash_with_esptool,
    list_serial_ports,
    pick_serial_port_interactive,
    run_full_flash,
    run_update_only,
)

# Override defaults, e.g. DeployConfig(circuitpy_root=Path("/media/youruser/CIRCUITPY"))  # Linux CIRCUITPY path
CFG = DeployConfig()
CFG

DeployConfig(serial_port='/dev/tty.usbmodem101', circuitpy_root=PosixPath('/Volumes/CIRCUITPY'), circuitpython_bin=PosixPath('/Users/silvioheinze/coding/firmware/deploy/bin/adafruit-circuitpython-espressif_esp32s3_devkitc_1_n8r8-de_DE.bin'), circuitpython_board_id='espressif_esp32s3_devkitc_1_n8r8', circuitpython_locale='de_DE', circuitpython_download_fallback_url='https://downloads.circuitpython.org/bin/espressif_esp32s3_devkitc_1_n8r8/de_DE/adafruit-circuitpython-espressif_esp32s3_devkitc_1_n8r8-de_DE-10.1.4.bin', settings_slot=0, do_erase_flash=True, do_write_firmware=True, wait_for_circuitpy_mount=True, wait_timeout_s=120.0, poll_interval_s=1.0, do_copy_firmware_tree=True, full_flash_settings=PosixPath('/Users/silvioheinze/coding/firmware/deploy/settings.toml'))

### Serial port

Run the next cell to **probe USB serial ports** (requires `pyserial`). It **bootstraps `deploy/`** like the setup cell, so it still works after a **kernel restart** without re-running setup. Type a **number** and Enter to set **`CFG.serial_port`**, or **Enter alone** to keep the current value (default **`/dev/tty.usbmodem101`**). If **`CFG`** does not exist yet, the cell creates it.

In [2]:
import sys
from pathlib import Path

_repo = Path.cwd().resolve()
_deploy_candidate = _repo if (_repo / "utils.py").is_file() else _repo / "deploy"
if str(_deploy_candidate) not in sys.path:
    sys.path.insert(0, str(_deploy_candidate))

import notebook_env

notebook_env.activate()

from utils import DeployConfig, pick_serial_port_interactive

if "CFG" not in globals():
    CFG = DeployConfig()

pick_serial_port_interactive(CFG)
CFG

Available serial ports:
  1) /dev/cu.debug-console — 'n/a'
  2) /dev/cu.Bluetooth-Incoming-Port — 'n/a'
  3) /dev/cu.BoseQuietControl30 — 'n/a'
  4) /dev/cu.usbmodem2101 — 'USB JTAG/serial debug unit'
Using serial_port: /dev/cu.usbmodem2101


DeployConfig(serial_port='/dev/cu.usbmodem2101', circuitpy_root=PosixPath('/Volumes/CIRCUITPY'), circuitpython_bin=PosixPath('/Users/silvioheinze/coding/firmware/deploy/bin/adafruit-circuitpython-espressif_esp32s3_devkitc_1_n8r8-de_DE.bin'), circuitpython_board_id='espressif_esp32s3_devkitc_1_n8r8', circuitpython_locale='de_DE', circuitpython_download_fallback_url='https://downloads.circuitpython.org/bin/espressif_esp32s3_devkitc_1_n8r8/de_DE/adafruit-circuitpython-espressif_esp32s3_devkitc_1_n8r8-de_DE-10.1.4.bin', settings_slot=0, do_erase_flash=True, do_write_firmware=True, wait_for_circuitpy_mount=True, wait_timeout_s=120.0, poll_interval_s=1.0, do_copy_firmware_tree=True, full_flash_settings=PosixPath('/Users/silvioheinze/coding/firmware/deploy/settings.toml'))

## Step 1 — Flash the board

Serial flashing only (**`erase_flash`** if enabled, then **`write_flash`**). Resolves a **`.bin`** via **`ensure_circuitpython_bin`** (existing file, `deploy/bin/`, download index, or fallback). Does **not** copy the repo onto **`CIRCUITPY`**.

Use **BOOT + RESET** on ESP32-S3 if the port does not appear. After this step, reset the board and wait until **`CIRCUITPY`** mounts before Step 2.

In [3]:
flash_with_esptool(CFG)

$ /Users/silvioheinze/coding/firmware/.venv/bin/python -m esptool --port /dev/cu.usbmodem2101 erase_flash
esptool v5.2.0
Serial port /dev/cu.usbmodem2101:
Connecting....
Detecting chip type... ESP32-S3
Connected to ESP32-S3 on /dev/cu.usbmodem2101:
Chip type:          ESP32-S3 (QFN56) (revision v0.2)
Features:           Wi-Fi, BT 5 (LE), Dual Core + LP Core, 240MHz, Embedded PSRAM 8MB (AP_3v3)
Crystal frequency:  40MHz
USB mode:           USB-Serial/JTAG
MAC:                d8:3b:da:43:d4:10

Uploading stub flasher...
Running stub flasher...
Stub flasher running.

Erasing flash memory (this may take a while)...
Flash memory erased successfully in 1.7 seconds.

Hard resetting via RTS pin...
Could not list download index (HTTP Error 404: Not Found); trying fallback URL.
Downloaded https://downloads.circuitpython.org/bin/espressif_esp32s3_devkitc_1_n8r8/de_DE/adafruit-circuitpython-espressif_esp32s3_devkitc_1_n8r8-de_DE-10.1.4.bin -> /Users/silvioheinze/coding/firmware/deploy/bin/adafruit

## Step 2 — Copy firmware to a new board

For a **new** board (or right after a full flash) when you want the repo **`firmware/`** tree plus **`deploy/settings.toml`** on **`CIRCUITPY`**. Waits for **`CFG.circuitpy_root`**, then copies.

Do **not** run Step 3 on the same device in the same session unless you intend the backup/slot workflow (Step 3 overwrites the tree then restores slot settings).

In [4]:
copy_firmware_to_circuitpy(CFG)

Found: /Volumes/CIRCUITPY
Copied firmware tree /Users/silvioheinze/coding/firmware/firmware -> /Volumes/CIRCUITPY
Copied /Users/silvioheinze/coding/firmware/deploy/settings.toml -> /Volumes/CIRCUITPY/settings.toml


## Step 3 — Update firmware (existing board)

For a board **already in use**: refresh the **`firmware/`** tree from the repo while keeping WiFi and other settings via **`settings_slot`** and **`deploy/settings_backups/slot_N/`**. **`CIRCUITPY`** must already be mounted. **No** esptool.

Skip Steps 1–2 when you only need an application update.

In [ ]:
run_update_only(CFG)

## Optional — List serial ports

Requires **`pyserial`**. Use the printed device path for **`CFG.serial_port`**.

In [ ]:
list_serial_ports()

## Optional — Shortcut for a new board

**`run_full_flash(CFG)`** runs **Step 1** then **Step 2** in one call (not Step 3). Use this **instead** of running the Step 1 and Step 2 cells above; leave the next line commented if you already use those cells, or you will flash twice.

In [ ]:
# run_full_flash(CFG)  # uncomment only if you skip Step 1 and Step 2 cells above